# Script 05 · Exportación, Assets y aplicación

**Curso:** Introducción a Google Earth Engine  
**Autora:** Grettel Vargas Azofeifa  
**Modalidad:** material de apoyo para GitHub y Google Earth Engine

> Los bloques de código están escritos en JavaScript para ejecutarse en el Editor de código de Google Earth Engine.

# Script 05 — Exportación, Assets y aplicación

Exporte resultados, incorpore datos propios, administre y comparta Assets, y construya una aplicación sencilla en Google Earth Engine.

## 🎯 Objetivos

1. Exportar imágenes y tablas desde Google Earth Engine.
2. Diferenciar entre Google Drive, Cloud Storage y Earth Engine Assets.
3. Subir datos vectoriales y ráster propios.
4. Subir datos propios y utilizarlos como Earth Engine Assets.
5. Compartir Assets y scripts con compañeros.
6. Crear y publicar una aplicación sencilla con título, descripción, mapa y selector de índice.

## 1. Preparar el análisis antes de exportar

Antes de configurar las exportaciones y construir la aplicación, ejecute este bloque para definir los parámetros, seleccionar las provincias, cargar Sentinel-2, calcular los índices y visualizar los resultados.

> **Práctica:** Secuencia recomendada: primero ejecute este código y revise los índices en el mapa. Después continúe con las secciones de exportación a Google Drive, exportación de tablas y exportación como Earth Engine Asset.

In [ ]:
//==================================================
// SCRIPT 05
// Exportación, Assets y aplicación
//==================================================

// 1. PARÁMETROS GENERALES
var nombresProvincias = ['Guanacaste', 'Puntarenas'];
var fechaInicio = '2024-01-01';
var fechaFin = '2024-03-31';
var nubosidadMaxima = 30;

// 2. ÁREA DE INTERÉS
var provincias = ee.FeatureCollection('FAO/GAUL/2015/level1');
var areaInteres = provincias
  .filter(ee.Filter.eq('ADM0_NAME', 'Costa Rica'))
  .filter(ee.Filter.inList('ADM1_NAME', nombresProvincias));

Map.centerObject(areaInteres, 8);
Map.addLayer(areaInteres, {color: 'yellow'}, 'Área de interés');

// 3. SENTINEL-2
var sentinel = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
  .filterBounds(areaInteres)
  .filterDate(fechaInicio, fechaFin)
  .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', nubosidadMaxima))
  .select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']);

var imagenSentinel = sentinel.median().clip(areaInteres);

// 4. ÍNDICES
var ndvi = imagenSentinel.normalizedDifference(['B8', 'B4']).rename('NDVI');

var evi = imagenSentinel.expression(
  '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
  {
    NIR: imagenSentinel.select('B8'),
    RED: imagenSentinel.select('B4'),
    BLUE: imagenSentinel.select('B2')
  }
).rename('EVI');

var ndwi = imagenSentinel.normalizedDifference(['B3', 'B8']).rename('NDWI');

var nbr = imagenSentinel.normalizedDifference(['B8', 'B12']).rename('NBR');

// 5. VISUALIZACIÓN
var visNDVI = {
  min: -1,
  max: 1,
  palette: ['ffffff', 'ffff00', '90ee90', '008000', '006400']
};

var visEVI = {
  min: -1,
  max: 1,
  palette: ['a52a2a', 'ffff00', '90ee90', '008000', '006400']
};

var visNDWI = {
  min: -1,
  max: 1,
  palette: ['a52a2a', 'ffff00', 'ffffff', '00ffff', '0000ff', '00008b']
};

var visNBR = {
  min: -1,
  max: 1,
  palette: [
    '000000',
    '4d0000',
    '800000',
    'b30000',
    'e60000',
    'ff3300',
    'ff9900',
    'ffff00'
  ]
};

Map.addLayer(ndvi, visNDVI, 'NDVI');
Map.addLayer(evi, visEVI, 'EVI', false);
Map.addLayer(ndwi, visNDWI, 'NDWI', false);
Map.addLayer(nbr, visNBR, 'NBR', false);


> **Nota:** Puede modificar: las provincias, las fechas y el porcentaje máximo de nubosidad desde el bloque de parámetros generales.

## 2. Opciones de exportación

Una vez obtenido un resultado en Google Earth Engine, puede guardarlo, descargarlo o mantenerlo dentro de la plataforma. La opción más adecuada depende de cómo utilizará posteriormente la información.

> **Práctica:** Antes de elegir: identifique si necesita descargar el resultado, reutilizarlo dentro de Earth Engine o integrarlo en un flujo de trabajo en la nube.

## 3. Exportar una imagen a Google Drive

En este ejemplo se exporta el NDVI como un archivo GeoTIFF.

In [ ]:
Export.image.toDrive({
  image: ndvi,
  description: 'NDVI_Guanacaste_2024',
  folder: 'Curso_GEE',
  fileNamePrefix: 'NDVI_Guanacaste_2024',
  region: areaInteres,
  scale: 10,
  crs: 'EPSG:3857',
  maxPixels: 1e13,
  fileFormat: 'GeoTIFF'
});


## 4. Exportar una tabla

Las capas vectoriales pueden exportarse como CSV, GeoJSON, KML, KMZ o Shapefile.

In [ ]:
Export.table.toDrive({
  collection: areaInteres,
  description: 'Area_Interes',
  folder: 'Curso_GEE',
  fileNamePrefix: 'Area_Interes',
  fileFormat: 'SHP'
});


## 5. Exportar una imagen como Earth Engine Asset

Esta opción permite guardar el resultado dentro de GEE para reutilizarlo en otros scripts o compartirlo con compañeros.

In [ ]:
Export.image.toAsset({
  image: ndvi,
  description: 'exportar_ndvi_como_asset',
  assetId: 'projects/mi-proyecto/assets/NDVI_Guanacaste_2024',
  region: areaInteres,
  scale: 10,
  crs: 'EPSG:3857',
  maxPixels: 1e13,
  pyramidingPolicy: {
    '.default': 'mean'
  }
});


> **Nota:** Importante: reemplace projects/mi-proyecto/assets/ por la ruta real de su proyecto de Earth Engine.

### Usar el Asset en otro script

In [ ]:
var ndviCompartido = ee.Image(
  'projects/mi-proyecto/assets/NDVI_Guanacaste_2024'
);

Map.addLayer(
  ndviCompartido,
  {min: -1, max: 1, palette: ['ffffff', 'ffff00', '008000']},
  'NDVI compartido'
);


## 6. Configurar la ventana de exportación

Después de ejecutar el código, la tarea aparece en la pestaña Tasks . Al seleccionar RUN , se muestra una ventana para revisar y confirmar la configuración.

> **Práctica:** Antes de ejecutar: revise el nombre, la ruta, la resolución, el CRS y el área de exportación.

## 7. Subir datos propios

Un Earth Engine Asset puede generarse mediante una exportación o crearse a partir de un archivo propio. En esta sección incorporará información vectorial o ráster para reutilizarla en scripts y aplicaciones.

> **Práctica:** Relación con la sección anterior: tanto una imagen exportada como un archivo cargado desde su computadora pueden almacenarse como Assets y después compartirse.

1. Abra la pestaña Assets .
2. Seleccione NEW .
3. Elija Table upload para datos vectoriales o Image upload para datos ráster.
4. Seleccione el archivo y defina el nombre del Asset.
5. Inicie la carga y espere a que finalice el proceso de ingestión.

### Importar una capa propia

In [ ]:
var fincas = ee.FeatureCollection(
  'projects/mi-proyecto/assets/Fincas'
);

Map.centerObject(fincas, 10);
Map.addLayer(fincas, {color: 'yellow'}, 'Fincas');


## 8. Compartir Assets con compañeros

Un Asset puede compartirse sin enviar archivos por correo. El otro usuario podrá utilizar la misma ruta dentro de sus scripts.

1. Abra la pestaña Assets .
2. Seleccione el Asset o la carpeta que desea compartir.
3. Abra la opción Share .
4. Agregue el correo del compañero.
5. Asigne el permiso correspondiente.

> **Nota:** Recomendación: comparta una carpeta cuando varios Assets forman parte del mismo proyecto. Esto facilita la administración del acceso.

## 9. Compartir scripts

Los scripts también pueden compartirse para que otros usuarios revisen o editen el código.

1. Guarde el script dentro de un repositorio.
2. Seleccione la opción Share .
3. Agregue el correo del compañero.
4. Defina si tendrá permiso de lectura o edición.

## 10. Crear una aplicación interactiva

La aplicación final permite escribir el nombre del autor, seleccionar una o varias provincias, visualizar uno o varios índices y hacer clic en un punto del mapa para generar una gráfica temporal de cada índice.

> **Nota:** Flujo de uso: escriba su nombre, seleccione las provincias y los índices, pulse Actualizar mapa y haga clic dentro del área seleccionada. El nombre aparecerá como autor en el mapa y las gráficas se mostrarán a la derecha.

In [ ]:
//==============================================================
// SCRIPT 05
// VISOR DE ÍNDICES ESPECTRALES DE COSTA RICA
//==============================================================
// 1. PARÁMETROS GENERALES
var fechaInicio = '2024-01-01';
var fechaFin = '2024-03-31';
var nubosidadMaxima = 30;
// 2. PROVINCIAS DE COSTA RICA
var provincias = ee.FeatureCollection(
  'FAO/GAUL/2015/level1'
).filter(
  ee.Filter.eq('ADM0_NAME', 'Costa Rica')
);
var nombresProvincias = [
  'Alajuela',
  'Cartago',
  'Guanacaste',
  'Heredia',
  'Limón',
  'Puntarenas',
  'San José'
];
// 3. MAPA
var mapa = ui.Map();
mapa.setOptions('HYBRID');
mapa.setCenter(-84.1, 9.8, 7);
mapa.style().set({stretch: 'both'});
// 4. PANEL DE CONTROLES
var panel = ui.Panel({
  style: {
    width: '320px',
    padding: '12px',
    stretch: 'vertical'
  }
});
// 5. TÍTULO E INFORMACIÓN
var titulo = ui.Label({
  value: 'Visor de índices espectrales de Costa Rica',
  style: {
    fontSize: '20px',
    fontWeight: 'bold',
    color: '#1565C0',
    whiteSpace: 'normal'
  }
});

panel.add(titulo);

// Nombre del autor.
panel.add(
  ui.Label({
    value: 'Autor de la aplicación',
    style: {
      fontWeight: 'bold',
      margin: '12px 0 5px 0'
    }
  })
);

var nombreAutor = ui.Textbox({
  placeholder: 'Escriba su nombre',
  value: '',
  style: {
    stretch: 'horizontal'
  }
});

panel.add(nombreAutor);

var descripcion = ui.Label({
  value:
    'Seleccione las provincias y los índices. ' +
    'Luego presione Actualizar mapa y haga clic ' +
    'sobre un punto para generar las gráficas.',
  style: {
    fontSize: '12px',
    whiteSpace: 'normal',
    margin: '12px 0'
  }
});

panel.add(descripcion);

// Mostrar el nombre del autor en el mapa.
var etiquetaAutor = ui.Label({
  value: 'Autor: sin especificar',
  style: {
    position: 'bottom-left',
    backgroundColor: 'white',
    padding: '8px',
    margin: '10px',
    fontWeight: 'bold'
  }
});

mapa.add(etiquetaAutor);
// 6. SELECCIÓN DE PROVINCIAS
panel.add(
  ui.Label({
    value: 'Provincias',
    style: {
      fontWeight: 'bold',
      margin: '12px 0 5px 0'
    }
  })
);
var checkboxesProvincias = {};
nombresProvincias.forEach(function(nombre) {
  var seleccionado =
    nombre === 'Guanacaste' ||
    nombre === 'Puntarenas';
  var checkbox = ui.Checkbox({
    label: nombre,
    value: seleccionado
  });
  checkboxesProvincias[nombre] = checkbox;
  panel.add(checkbox);
});
// 7. SELECCIÓN DE ÍNDICES
panel.add(
  ui.Label({
    value: 'Índices espectrales',
    style: {
      fontWeight: 'bold',
      margin: '15px 0 5px 0'
    }
  })
);
var nombresIndices = ['NDVI', 'EVI', 'NDWI', 'NBR'];
var checkboxesIndices = {};
nombresIndices.forEach(function(nombre) {
  var checkbox = ui.Checkbox({
    label: nombre,
    value: nombre === 'NDVI'
  });
  checkboxesIndices[nombre] = checkbox;
  panel.add(checkbox);
});
// 8. LEER LAS SELECCIONES
function obtenerProvinciasSeleccionadas() {
  var seleccionadas = [];
  nombresProvincias.forEach(function(nombre) {
    if (checkboxesProvincias[nombre].getValue()) {
      seleccionadas.push(nombre);
    }
  });
  return seleccionadas;
}
function obtenerIndicesSeleccionados() {
  var seleccionados = [];
  nombresIndices.forEach(function(nombre) {
    if (checkboxesIndices[nombre].getValue()) {
      seleccionados.push(nombre);
    }
  });
  return seleccionados;
}
// 9. PALETAS
var visualizaciones = {
  NDVI: {
    min: -1,
    max: 1,
    palette: [
      'ffffff',
      'ffff00',
      '90ee90',
      '008000',
      '006400'
    ]
  },
  EVI: {
    min: -1,
    max: 1,
    palette: [
      'a52a2a',
      'ffff00',
      '90ee90',
      '008000',
      '006400'
    ]
  },
  NDWI: {
    min: -1,
    max: 1,
    palette: [
      'a52a2a',
      'ffff00',
      'ffffff',
      '00ffff',
      '0000ff',
      '00008b'
    ]
  },
  NBR: {
    min: -1,
    max: 1,
    palette: [
      '000000',
      '4d0000',
      '800000',
      'b30000',
      'e60000',
      'ff3300',
      'ff9900',
      'ffff00'
    ]
  }
};
// 10. CALCULAR LOS ÍNDICES
function calcularIndices(imagen) {
  var reflectancia = imagen.multiply(0.0001);
  var ndvi = reflectancia
    .normalizedDifference(['B8', 'B4'])
    .rename('NDVI');
  var evi = reflectancia.expression(
    '2.5 * ((NIR - RED) / ' +
    '(NIR + 6 * RED - 7.5 * BLUE + 1))',
    {
      NIR: reflectancia.select('B8'),
      RED: reflectancia.select('B4'),
      BLUE: reflectancia.select('B2')
    }
  ).rename('EVI');
  var ndwi = reflectancia
    .normalizedDifference(['B3', 'B8'])
    .rename('NDWI');
  var nbr = reflectancia
    .normalizedDifference(['B8', 'B12'])
    .rename('NBR');
  return reflectancia
    .addBands([ndvi, evi, ndwi, nbr])
    .copyProperties(imagen, ['system:time_start']);
}
// 11. VARIABLES DE LA APLICACIÓN
var areaInteres;
var coleccionIndices;
var imagenIndices;
var capaPunto;
var indicesSeleccionadosActuales = [];
// 12. PANEL DE GRÁFICAS
var panelGraficas = ui.Panel({
  layout: ui.Panel.Layout.flow('vertical'),
  style: {
    width: '400px',
    padding: '10px',
    stretch: 'vertical'
  }
});
function mostrarMensajeGraficas(mensaje, color) {
  panelGraficas.clear();
  panelGraficas.add(
    ui.Label({
      value: 'Gráficas de los índices',
      style: {
        fontSize: '17px',
        fontWeight: 'bold',
        color: '#1565C0'
      }
    })
  );
  panelGraficas.add(
    ui.Label({
      value: mensaje,
      style: {
        color: color || '#555555',
        whiteSpace: 'normal',
        margin: '8px 0'
      }
    })
  );
}
mostrarMensajeGraficas(
  'Haga clic en un punto del mapa para generar las gráficas.'
);
// 13. COLECCIÓN SENTINEL-2
function crearColeccionSentinel(area) {
  return ee.ImageCollection(
    'COPERNICUS/S2_SR_HARMONIZED'
  )
  .filterBounds(area)
  .filterDate(fechaInicio, fechaFin)
  .filter(
    ee.Filter.lte(
      'CLOUDY_PIXEL_PERCENTAGE',
      nubosidadMaxima
    )
  )
  .select([
    'B2',
    'B3',
    'B4',
    'B8',
    'B11',
    'B12'
  ])
  .map(calcularIndices);
}
// 14. ACTUALIZAR EL MAPA
function actualizarMapa() {

  // Leer el nombre escrito por el estudiante.
  var autor = nombreAutor.getValue();

  // Mostrar un texto predeterminado si el cuadro está vacío.
  if (!autor) {
    autor = 'sin especificar';
  }

  // Actualizar la etiqueta del autor en el mapa.
  etiquetaAutor.setValue(
    'Autor: ' + autor
  );

  var provinciasSeleccionadas =
    obtenerProvinciasSeleccionadas();
  var indicesSeleccionados =
    obtenerIndicesSeleccionados();
  if (provinciasSeleccionadas.length === 0) {
    mostrarMensajeGraficas(
      'Seleccione al menos una provincia.',
      'red'
    );
    return;
  }
  if (indicesSeleccionados.length === 0) {
    mostrarMensajeGraficas(
      'Seleccione al menos un índice.',
      'red'
    );
    return;
  }
  indicesSeleccionadosActuales =
    indicesSeleccionados;
  areaInteres = provincias.filter(
    ee.Filter.inList(
      'ADM1_NAME',
      provinciasSeleccionadas
    )
  );
  coleccionIndices =
    crearColeccionSentinel(areaInteres);
  imagenIndices = coleccionIndices
    .median()
    .clip(areaInteres);
  mapa.layers().reset();
  mapa.addLayer(
    areaInteres.style({
      color: 'yellow',
      fillColor: '00000000',
      width: 2
    }),
    {},
    'Área de interés'
  );
  indicesSeleccionados.forEach(
    function(indice, posicion) {
      mapa.addLayer(
        imagenIndices.select(indice),
        visualizaciones[indice],
        indice,
        posicion === 0
      );
    }
  );
  mapa.centerObject(
    areaInteres,
    provinciasSeleccionadas.length === 1 ? 9 : 8
  );
  capaPunto = null;
  mostrarMensajeGraficas(
    'Haga clic en un punto del mapa para generar las gráficas.'
  );
}
// 15. BOTÓN ACTUALIZAR
var botonActualizar = ui.Button({
  label: 'Actualizar mapa',
  onClick: actualizarMapa,
  style: {
    stretch: 'horizontal',
    margin: '12px 0',
    fontWeight: 'bold'
  }
});
panel.add(botonActualizar);
// 16. INSTRUCCIONES
panel.add(
  ui.Label({
    value: 'Consultar un punto',
    style: {
      fontWeight: 'bold',
      margin: '15px 0 5px 0'
    }
  })
);
panel.add(
  ui.Label({
    value:
      'Haga clic dentro de las provincias seleccionadas. ' +
      'Las gráficas aparecerán en el panel derecho.',
    style: {
      fontSize: '12px',
      whiteSpace: 'normal'
    }
  })
);
// 17. GENERAR GRÁFICAS AL HACER CLIC
mapa.onClick(function(coordenadas) {
  if (!coleccionIndices) {
    mostrarMensajeGraficas(
      'Primero seleccione las provincias y los índices.',
      'red'
    );
    return;
  }
  panelGraficas.clear();
  var punto = ee.Geometry.Point([
    coordenadas.lon,
    coordenadas.lat
  ]);
  if (capaPunto) {
    mapa.layers().remove(capaPunto);
  }
  capaPunto = ui.Map.Layer(
    punto,
    {color: 'red'},
    'Punto seleccionado'
  );
  mapa.layers().add(capaPunto);
  panelGraficas.add(
    ui.Label({
      value: 'Gráficas de los índices',
      style: {
        fontSize: '17px',
        fontWeight: 'bold',
        color: '#1565C0'
      }
    })
  );
  panelGraficas.add(
    ui.Label({
      value:
        'Punto: ' +
        coordenadas.lat.toFixed(5) +
        ', ' +
        coordenadas.lon.toFixed(5),
      style: {
        fontWeight: 'bold',
        margin: '8px 0'
      }
    })
  );
  indicesSeleccionadosActuales.forEach(
    function(indice) {
      var grafica = ui.Chart.image.series({
        imageCollection:
          coleccionIndices.select(indice),
        region:
          punto.buffer(20),
        reducer:
          ee.Reducer.mean(),
        scale:
          10,
        xProperty:
          'system:time_start'
      })
      .setOptions({
        title:
          'Serie temporal de ' + indice,
        hAxis: {
          title: 'Fecha',
          format: 'dd-MM-yyyy'
        },
        vAxis: {
          title: indice,
          viewWindow: {
            min: -1,
            max: 1
          }
        },
        lineWidth: 2,
        pointSize: 4,
        interpolateNulls: true,
        legend: {position: 'none'},
        chartArea: {
          left: 55,
          right: 15,
          top: 45,
          bottom: 45
        }
      });
      panelGraficas.add(grafica);
    }
  );
});
// 18. MOSTRAR LA APLICACIÓN
ui.root.clear();
ui.root.setLayout(
  ui.Panel.Layout.flow('horizontal')
);
ui.root.add(panel);
ui.root.add(mapa);
ui.root.add(panelGraficas);
// 19. CONFIGURACIÓN INICIAL
actualizarMapa();


> **Práctica:** Resultado esperado: al marcar varias provincias y varios índices, el mapa muestra cada índice recortado al conjunto de provincias. Al dibujar un punto o polígono, el panel inferior genera una gráfica temporal independiente para cada índice seleccionado.

## 11. Publicar y compartir la aplicación

1. Guarde el script de la aplicación.
2. Seleccione Apps en el Code Editor.
3. Elija NEW APP .
4. Seleccione el script correspondiente.
5. Defina el nombre, el título y la descripción de la aplicación.
6. Revise los permisos de los Assets utilizados.
7. Publique la aplicación y comparta su enlace.

> **Nota:** Importante: los usuarios de la aplicación deben tener acceso a los Assets privados utilizados. Para una aplicación pública, revise cuidadosamente los permisos de los datos.

## 🧩 Ejercicio integrador

Cada estudiante desarrollará y publicará una aplicación personalizada.

1. Escriba su nombre como autor de la aplicación.
2. Seleccione una o varias provincias.
3. Seleccione uno o varios índices: NDVI, EVI, NDWI y NBR.
4. Presione Actualizar mapa.
5. Compruebe que su nombre aparece en la esquina inferior izquierda del mapa.
6. Haga clic en un punto dentro del área seleccionada.
7. Observe las gráficas temporales en el panel derecho.
8. Comparta el script o publique la aplicación.

> **Práctica:** Producto final: una aplicación introductoria personalizada con nombre del autor, selección de provincias e índices, mapa central y gráficas temporales.

## 12. Código completo del ejercicio

Este bloque contiene la aplicación completa y simplificada. Puede copiarlo directamente en el Code Editor de Google Earth Engine.

> **Nota:** Uso: escriba el nombre del autor, seleccione las provincias y los índices, pulse Actualizar mapa y haga clic en un punto. El autor aparecerá en el mapa y las gráficas en el panel derecho.

In [ ]:
//==============================================================
// SCRIPT 05
// VISOR DE ÍNDICES ESPECTRALES DE COSTA RICA
//==============================================================
// 1. PARÁMETROS GENERALES
var fechaInicio = '2024-01-01';
var fechaFin = '2024-03-31';
var nubosidadMaxima = 30;
// 2. PROVINCIAS DE COSTA RICA
var provincias = ee.FeatureCollection(
  'FAO/GAUL/2015/level1'
).filter(
  ee.Filter.eq('ADM0_NAME', 'Costa Rica')
);
var nombresProvincias = [
  'Alajuela',
  'Cartago',
  'Guanacaste',
  'Heredia',
  'Limón',
  'Puntarenas',
  'San José'
];
// 3. MAPA
var mapa = ui.Map();
mapa.setOptions('HYBRID');
mapa.setCenter(-84.1, 9.8, 7);
mapa.style().set({stretch: 'both'});
// 4. PANEL DE CONTROLES
var panel = ui.Panel({
  style: {
    width: '320px',
    padding: '12px',
    stretch: 'vertical'
  }
});
// 5. TÍTULO E INFORMACIÓN
var titulo = ui.Label({
  value: 'Visor de índices espectrales de Costa Rica',
  style: {
    fontSize: '20px',
    fontWeight: 'bold',
    color: '#1565C0',
    whiteSpace: 'normal'
  }
});

panel.add(titulo);

// Nombre del autor.
panel.add(
  ui.Label({
    value: 'Autor de la aplicación',
    style: {
      fontWeight: 'bold',
      margin: '12px 0 5px 0'
    }
  })
);

var nombreAutor = ui.Textbox({
  placeholder: 'Escriba su nombre',
  value: '',
  style: {
    stretch: 'horizontal'
  }
});

panel.add(nombreAutor);

var descripcion = ui.Label({
  value:
    'Seleccione las provincias y los índices. ' +
    'Luego presione Actualizar mapa y haga clic ' +
    'sobre un punto para generar las gráficas.',
  style: {
    fontSize: '12px',
    whiteSpace: 'normal',
    margin: '12px 0'
  }
});

panel.add(descripcion);

// Mostrar el nombre del autor en el mapa.
var etiquetaAutor = ui.Label({
  value: 'Autor: sin especificar',
  style: {
    position: 'bottom-left',
    backgroundColor: 'white',
    padding: '8px',
    margin: '10px',
    fontWeight: 'bold'
  }
});

mapa.add(etiquetaAutor);
// 6. SELECCIÓN DE PROVINCIAS
panel.add(
  ui.Label({
    value: 'Provincias',
    style: {
      fontWeight: 'bold',
      margin: '12px 0 5px 0'
    }
  })
);
var checkboxesProvincias = {};
nombresProvincias.forEach(function(nombre) {
  var seleccionado =
    nombre === 'Guanacaste' ||
    nombre === 'Puntarenas';
  var checkbox = ui.Checkbox({
    label: nombre,
    value: seleccionado
  });
  checkboxesProvincias[nombre] = checkbox;
  panel.add(checkbox);
});
// 7. SELECCIÓN DE ÍNDICES
panel.add(
  ui.Label({
    value: 'Índices espectrales',
    style: {
      fontWeight: 'bold',
      margin: '15px 0 5px 0'
    }
  })
);
var nombresIndices = ['NDVI', 'EVI', 'NDWI', 'NBR'];
var checkboxesIndices = {};
nombresIndices.forEach(function(nombre) {
  var checkbox = ui.Checkbox({
    label: nombre,
    value: nombre === 'NDVI'
  });
  checkboxesIndices[nombre] = checkbox;
  panel.add(checkbox);
});
// 8. LEER LAS SELECCIONES
function obtenerProvinciasSeleccionadas() {
  var seleccionadas = [];
  nombresProvincias.forEach(function(nombre) {
    if (checkboxesProvincias[nombre].getValue()) {
      seleccionadas.push(nombre);
    }
  });
  return seleccionadas;
}
function obtenerIndicesSeleccionados() {
  var seleccionados = [];
  nombresIndices.forEach(function(nombre) {
    if (checkboxesIndices[nombre].getValue()) {
      seleccionados.push(nombre);
    }
  });
  return seleccionados;
}
// 9. PALETAS
var visualizaciones = {
  NDVI: {
    min: -1,
    max: 1,
    palette: [
      'ffffff',
      'ffff00',
      '90ee90',
      '008000',
      '006400'
    ]
  },
  EVI: {
    min: -1,
    max: 1,
    palette: [
      'a52a2a',
      'ffff00',
      '90ee90',
      '008000',
      '006400'
    ]
  },
  NDWI: {
    min: -1,
    max: 1,
    palette: [
      'a52a2a',
      'ffff00',
      'ffffff',
      '00ffff',
      '0000ff',
      '00008b'
    ]
  },
  NBR: {
    min: -1,
    max: 1,
    palette: [
      '000000',
      '4d0000',
      '800000',
      'b30000',
      'e60000',
      'ff3300',
      'ff9900',
      'ffff00'
    ]
  }
};
// 10. CALCULAR LOS ÍNDICES
function calcularIndices(imagen) {
  var reflectancia = imagen.multiply(0.0001);
  var ndvi = reflectancia
    .normalizedDifference(['B8', 'B4'])
    .rename('NDVI');
  var evi = reflectancia.expression(
    '2.5 * ((NIR - RED) / ' +
    '(NIR + 6 * RED - 7.5 * BLUE + 1))',
    {
      NIR: reflectancia.select('B8'),
      RED: reflectancia.select('B4'),
      BLUE: reflectancia.select('B2')
    }
  ).rename('EVI');
  var ndwi = reflectancia
    .normalizedDifference(['B3', 'B8'])
    .rename('NDWI');
  var nbr = reflectancia
    .normalizedDifference(['B8', 'B12'])
    .rename('NBR');
  return reflectancia
    .addBands([ndvi, evi, ndwi, nbr])
    .copyProperties(imagen, ['system:time_start']);
}
// 11. VARIABLES DE LA APLICACIÓN
var areaInteres;
var coleccionIndices;
var imagenIndices;
var capaPunto;
var indicesSeleccionadosActuales = [];
// 12. PANEL DE GRÁFICAS
var panelGraficas = ui.Panel({
  layout: ui.Panel.Layout.flow('vertical'),
  style: {
    width: '400px',
    padding: '10px',
    stretch: 'vertical'
  }
});
function mostrarMensajeGraficas(mensaje, color) {
  panelGraficas.clear();
  panelGraficas.add(
    ui.Label({
      value: 'Gráficas de los índices',
      style: {
        fontSize: '17px',
        fontWeight: 'bold',
        color: '#1565C0'
      }
    })
  );
  panelGraficas.add(
    ui.Label({
      value: mensaje,
      style: {
        color: color || '#555555',
        whiteSpace: 'normal',
        margin: '8px 0'
      }
    })
  );
}
mostrarMensajeGraficas(
  'Haga clic en un punto del mapa para generar las gráficas.'
);
// 13. COLECCIÓN SENTINEL-2
function crearColeccionSentinel(area) {
  return ee.ImageCollection(
    'COPERNICUS/S2_SR_HARMONIZED'
  )
  .filterBounds(area)
  .filterDate(fechaInicio, fechaFin)
  .filter(
    ee.Filter.lte(
      'CLOUDY_PIXEL_PERCENTAGE',
      nubosidadMaxima
    )
  )
  .select([
    'B2',
    'B3',
    'B4',
    'B8',
    'B11',
    'B12'
  ])
  .map(calcularIndices);
}
// 14. ACTUALIZAR EL MAPA
function actualizarMapa() {

  // Leer el nombre escrito por el estudiante.
  var autor = nombreAutor.getValue();

  // Mostrar un texto predeterminado si el cuadro está vacío.
  if (!autor) {
    autor = 'sin especificar';
  }

  // Actualizar la etiqueta del autor en el mapa.
  etiquetaAutor.setValue(
    'Autor: ' + autor
  );

  var provinciasSeleccionadas =
    obtenerProvinciasSeleccionadas();
  var indicesSeleccionados =
    obtenerIndicesSeleccionados();
  if (provinciasSeleccionadas.length === 0) {
    mostrarMensajeGraficas(
      'Seleccione al menos una provincia.',
      'red'
    );
    return;
  }
  if (indicesSeleccionados.length === 0) {
    mostrarMensajeGraficas(
      'Seleccione al menos un índice.',
      'red'
    );
    return;
  }
  indicesSeleccionadosActuales =
    indicesSeleccionados;
  areaInteres = provincias.filter(
    ee.Filter.inList(
      'ADM1_NAME',
      provinciasSeleccionadas
    )
  );
  coleccionIndices =
    crearColeccionSentinel(areaInteres);
  imagenIndices = coleccionIndices
    .median()
    .clip(areaInteres);
  mapa.layers().reset();
  mapa.addLayer(
    areaInteres.style({
      color: 'yellow',
      fillColor: '00000000',
      width: 2
    }),
    {},
    'Área de interés'
  );
  indicesSeleccionados.forEach(
    function(indice, posicion) {
      mapa.addLayer(
        imagenIndices.select(indice),
        visualizaciones[indice],
        indice,
        posicion === 0
      );
    }
  );
  mapa.centerObject(
    areaInteres,
    provinciasSeleccionadas.length === 1 ? 9 : 8
  );
  capaPunto = null;
  mostrarMensajeGraficas(
    'Haga clic en un punto del mapa para generar las gráficas.'
  );
}
// 15. BOTÓN ACTUALIZAR
var botonActualizar = ui.Button({
  label: 'Actualizar mapa',
  onClick: actualizarMapa,
  style: {
    stretch: 'horizontal',
    margin: '12px 0',
    fontWeight: 'bold'
  }
});
panel.add(botonActualizar);
// 16. INSTRUCCIONES
panel.add(
  ui.Label({
    value: 'Consultar un punto',
    style: {
      fontWeight: 'bold',
      margin: '15px 0 5px 0'
    }
  })
);
panel.add(
  ui.Label({
    value:
      'Haga clic dentro de las provincias seleccionadas. ' +
      'Las gráficas aparecerán en el panel derecho.',
    style: {
      fontSize: '12px',
      whiteSpace: 'normal'
    }
  })
);
// 17. GENERAR GRÁFICAS AL HACER CLIC
mapa.onClick(function(coordenadas) {
  if (!coleccionIndices) {
    mostrarMensajeGraficas(
      'Primero seleccione las provincias y los índices.',
      'red'
    );
    return;
  }
  panelGraficas.clear();
  var punto = ee.Geometry.Point([
    coordenadas.lon,
    coordenadas.lat
  ]);
  if (capaPunto) {
    mapa.layers().remove(capaPunto);
  }
  capaPunto = ui.Map.Layer(
    punto,
    {color: 'red'},
    'Punto seleccionado'
  );
  mapa.layers().add(capaPunto);
  panelGraficas.add(
    ui.Label({
      value: 'Gráficas de los índices',
      style: {
        fontSize: '17px',
        fontWeight: 'bold',
        color: '#1565C0'
      }
    })
  );
  panelGraficas.add(
    ui.Label({
      value:
        'Punto: ' +
        coordenadas.lat.toFixed(5) +
        ', ' +
        coordenadas.lon.toFixed(5),
      style: {
        fontWeight: 'bold',
        margin: '8px 0'
      }
    })
  );
  indicesSeleccionadosActuales.forEach(
    function(indice) {
      var grafica = ui.Chart.image.series({
        imageCollection:
          coleccionIndices.select(indice),
        region:
          punto.buffer(20),
        reducer:
          ee.Reducer.mean(),
        scale:
          10,
        xProperty:
          'system:time_start'
      })
      .setOptions({
        title:
          'Serie temporal de ' + indice,
        hAxis: {
          title: 'Fecha',
          format: 'dd-MM-yyyy'
        },
        vAxis: {
          title: indice,
          viewWindow: {
            min: -1,
            max: 1
          }
        },
        lineWidth: 2,
        pointSize: 4,
        interpolateNulls: true,
        legend: {position: 'none'},
        chartArea: {
          left: 55,
          right: 15,
          top: 45,
          bottom: 45
        }
      });
      panelGraficas.add(grafica);
    }
  );
});
// 18. MOSTRAR LA APLICACIÓN
ui.root.clear();
ui.root.setLayout(
  ui.Panel.Layout.flow('horizontal')
);
ui.root.add(panel);
ui.root.add(mapa);
ui.root.add(panelGraficas);
// 19. CONFIGURACIÓN INICIAL
actualizarMapa();


## 13. ✅ Lo aprendido

- Exportar imágenes y tablas.
- Elegir entre Drive, Cloud Storage y EE Asset.
- Configurar una tarea de exportación.
- Subir datos propios.
- Compartir Assets y scripts.
- Crear y publicar una aplicación sencilla.